<a href="https://colab.research.google.com/github/sumeyyeozturk0/Low-Light-OBJECT-detection/blob/main/derin%C3%B6%C4%9Frenme_%C3%B6dev2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Gerekli Kütüphanelerin Kurulumu
YOLOv8 mimarisini kullanabilmek için Ultralytics kütüphanesi çalışma ortamına yüklenmiştir.

In [ ]:
!pip install ultralytics


### 2. Google Drive Bağlantısı
Eğitim sırasında oluşacak model ağırlıklarını (weights) kaybetmemek ve veri setine erişmek için Google Drive ortamı Colab'e entegre edilmiştir.

In [ ]:
from google.colab import drive
#kaydedilmesi için drive'a bağlıyoruz
drive.mount('/content/drive')


### 3. YOLOv8 Modelinin Eğitimi
YOLOv8n (Nano) önceden eğitilmiş modeli yüklenmiş ve Pascal VOC veri seti kullanılarak 5 epoch boyunca eğitim gerçekleştirilmiştir. Sonuçlar Drive üzerine kaydedilmiştir.

In [ ]:
 from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# Eğitimi başlatıyoruz
results = model.train(data='VOC.yaml', epochs=5, imgsz=640, project='/content/drive/MyDrive/Derin_Ogrenme_Odevi')


### 4. Eğitilen Modelin Yüklenmesi ve Test Edilmesi
Eğitim sonucu elde edilen en iyi ağırlık dosyası (`best.pt`) yüklenmiş ve çeşitli ışık koşullarında (gündüz/gece) çekilen test fotoğrafları üzerinde %80 güven skoru (confidence threshold) ile tahminleme yapılmıştır.

In [ ]:
from ultralytics import YOLO

# Eğittiğimiz en iyi modelimizi yüklüyoruz
model = YOLO('/content/drive/MyDrive/Derin_Ogrenme_Odevi/train/weights/best.pt')

# conf=0.80 parametresi ile %80 sınırımızı koyduk. save=True ile sonuçları kaydediyoruz.

test_fotograflari = ['gunduz_1.jpeg', 'gunduz_2.jpeg', 'gunduz_3.jpeg', 'gunduz_4.jpeg', 'gece_fotografi.jpeg']

sonuclar = model.predict(source=test_fotograflari, conf=0.80, save=True)

### 5. Test Sonuçlarının Görüntülenmesi
Modelin tespit yapıp üzerine sınır kutuları çizdiği çıktılar ekrana yazdırılmıştır.

In [ ]:
import os
from IPython.display import Image, display

klasor = '/content/runs/detect/predict'

resimler = [f for f in os.listdir(klasor) if f.endswith(('.jpg', '.jpeg'))]

for resim in resimler:
    tam_yol = os.path.join(klasor, resim)
    display(Image(filename=tam_yol, width=300))

### 6. Web Arayüzü İçin Kütüphane Kurulumu
Modeli son kullanıcı ile buluşturacak etkileşimli web arayüzünü oluşturmak için Gradio kütüphanesi kurulmuştur.

In [ ]:
!pip install gradio


### 7. Deployment (Uygulama) ve Web Arayüzünün Başlatılması
Eğitilen model, Gradio kullanılarak bir web uygulamasına dönüştürülmüştür. Bu arayüz üzerinden kullanıcılar anlık olarak fotoğraf yükleyebilmekte ve tespit sonuçlarını görsel olarak alabilmektedir.

In [ ]:
import gradio as gr
from ultralytics import YOLO
import cv2

model = YOLO('/content/drive/MyDrive/Derin_Ogrenme_Odevi/train/weights/best.pt')

def nesne_tespiti(resim):

    sonuclar = model.predict(source=resim, conf=0.80)
    cizimli_resim = sonuclar[0].plot()
    cizimli_resim_rgb = cv2.cvtColor(cizimli_resim, cv2.COLOR_BGR2RGB)

    return cizimli_resim_rgb

# Web arayüzünün tasarımını yapıyoruz
arayuz = gr.Interface(
    fn=nesne_tespiti,
    inputs=gr.Image(type="numpy", label="Buraya Fotoğraf Yükleyiniz"),
    outputs=gr.Image(type="numpy", label="YOLOv8 Tespit Sonucu"),
    title="Nesne Tespiti Uygulaması",
    description="Gündüz veya gece çekilmiş bir fotoğrafı yükleyin. Model nesneleri tespit edip işaretleyecektir."
)
# Arayüzü internete açıyoruz
arayuz.launch(share=True)
